# The perceptron
> The OG neural net classifier without any bells and whistles.

References:
* https://en.wikipedia.org/wiki/Perceptron#Steps

Fun facts:
* Dates back to _McCullock and Pitts 1943, A logical calculus of the ideas immanent in nervous activity_
* Was only first mechanically implemented by _Rosenblatt 1953, The perceptron: A probabilistic model for information storage and organization in the brain_

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import sklearn.datasets as sk_datasets
from sklearn import metrics
from plotnine import (
    ggplot,
    aes,
    geom_tile,
    geom_point,
    geom_line,
    theme,
    labs,
    scale_x_continuous,
    scale_y_continuous,
)
import polars as pl
import random_neural_net_models.perceptron as perceptron

In [ ]:
rng = np.random.RandomState(42)

## Binary classification

In [ ]:
X, y = sk_datasets.make_blobs(
    n_samples=1_000,
    n_features=2,
    centers=2,
    random_state=rng,
)

In [ ]:
tmp = pl.DataFrame({"x0": X[:, 0], "x1": X[:, 1], "y": y})
(ggplot(tmp, aes("x0", "x1", color="y")) + geom_point())

In [ ]:
model = perceptron.PerceptronClassifier(epochs=10, verbose=True)

In [ ]:
model.fit(X, y)

In [ ]:
def plot_train_loss(model: perceptron.PerceptronClassifier) -> ggplot:
    tmp = pl.DataFrame({"iteration": range(model.epochs), "loss": model.errors_})
    return (
        ggplot(tmp, aes(x="iteration", y="loss"))
        + geom_line(alpha=0.3)
        + geom_point(size=3, alpha=0.3, fill="none")
        + labs(title="Training history: loss vs iteration", x="epoch")
    )


plot_train_loss(model)

In [ ]:
y_hat = model.predict(X)
y_hat[:5]

In [ ]:
print(metrics.classification_report(y, y_hat))

In [ ]:
x0 = np.linspace(X[:, 0].min(), X[:, 0].max(), 100)
x1 = np.linspace(X[:, 1].min(), X[:, 1].max(), 100)
X0, X1 = np.meshgrid(x0, x1)
X_plot = np.array([X0.ravel(), X1.ravel()]).T

In [ ]:
y_pred = model.predict(X_plot)

In [ ]:
def plot_mesh(
    X0: np.ndarray, X1: np.ndarray, y_prob: np.ndarray, y: np.ndarray
) -> ggplot:
    tmp_background = pl.DataFrame(
        {
            "x0": X0.ravel().tolist(),
            "x1": X1.ravel().tolist(),
            "y_prob": y_prob.ravel().tolist(),
        }
    )
    tmp_scatter = pl.DataFrame(
        {
            "x0": X[:, 0].ravel().tolist(),
            "x1": X[:, 1].ravel().tolist(),
            "y": [int(v) for v in (y)],
        }
    )

    xmin, xmax = X0.min(), X0.max()
    ymin, ymax = X1.min(), X1.max()
    return (
        ggplot(tmp_background, aes("x0", "x1", fill="y_prob"))
        + geom_tile(alpha=0.1)
        + geom_point(
            data=tmp_scatter, mapping=aes("x0", "x1", color="y"), inherit_aes=False
        )
        + labs(fill="P(y=1)", color="label")
        + theme(legend_position="right")
        + scale_x_continuous(limits=(xmin, xmax), expand=(0, 0))
        + scale_y_continuous(limits=(ymin, ymax), expand=(0, 0))
    )


plot_mesh(X0, X1, y_pred, y)

## Multi-class classification

In [ ]:
X, y = sk_datasets.make_blobs(
    n_samples=1_000,
    n_features=2,
    centers=3,
    random_state=rng,
)

In [ ]:
tmp = pl.DataFrame({"x0": X[:, 0], "x1": X[:, 1], "y": y})
(ggplot(tmp, aes("x0", "x1", color="y")) + geom_point())

In [ ]:
model = perceptron.PerceptronClassifier(epochs=10, verbose=True)

In [ ]:
model.fit(X, y)

In [ ]:
plot_train_loss(model)

In [ ]:
y_hat = model.predict(X)
y_hat[:5]

In [ ]:
print(metrics.classification_report(y, y_hat))

In [ ]:
X.shape

In [ ]:
x0 = np.linspace(X[:, 0].min(), X[:, 0].max(), 100)
x1 = np.linspace(X[:, 1].min(), X[:, 1].max(), 100)
X0, X1 = np.meshgrid(x0, x1)
X_plot = np.array([X0.ravel(), X1.ravel()]).T

In [ ]:
y_pred = model.predict(X_plot)

In [ ]:
plot_mesh(X0, X1, y_pred, y)